# Voice Replacer / AI Dubbing — Colab (free GPU)

Run the dubbing pipeline on Colab's **free GPU** — a 9-minute video takes
~5-10 min here instead of 1-2 hours on a CPU.

**Before you start:** `Runtime -> Change runtime type -> Hardware accelerator -> GPU -> Save`.

Then run each cell top to bottom (Shift+Enter).

## 1. Check the GPU is on

In [ ]:
!nvidia-smi -L || echo 'NO GPU: use Runtime -> Change runtime type -> GPU, then re-run.'

## 2. Download the project

This clones the public project repo — no login or token needed.

In [ ]:
import os

USER = 'PrashantRMarathe'
REPO = 'voice-replacer'

if os.path.isdir(REPO):
    # Auto-pull the latest fixes (tiny download) so Run all is always current.
    !cd {REPO} && git pull --quiet
else:
    !git clone https://github.com/{USER}/{REPO}.git
%cd {REPO}
print('Now in:', os.getcwd(), '(latest version)')

## 3. Install dependencies (~3-5 min, first run only)

In [ ]:
# Core: voice + transcription + translation
!pip install -q coqui-tts 'transformers<5' ffmpeg-python deep-translator num2words soundfile openai-whisper
# Keep original music/SFX under the new voice
!pip install -q demucs
# OPTIONAL alternate voice engine (often better accent/softness). Non-fatal.
!pip install -q f5-tts 2>/dev/null || echo "NOTE: f5-tts not installed (optional)."
print('Dependencies installed.')

## 4. Configure

On a GPU you can afford a bigger, more accurate transcription model. Edit the
settings below if you like, then run the cell.

In [ ]:
import os
os.environ['COQUI_TOS_AGREED'] = '1'  # accept XTTS non-commercial license

import config
config.WHISPER_MODEL = 'large-v3'     # best transcription on GPU
config.WHISPER_BACKEND = 'openai'     # avoids Colab PyAV issue

# ---- VOICE ENGINE ----------------------------------------------------------
# XTTS is the default. If the voice's accent/softness isn't matching, flip this
# to True to use F5-TTS instead (often more natural). Compare and keep the best.
USE_F5 = False
config.TTS_BACKEND = 'f5' if USE_F5 else 'xtts'

# ---- Keep the voice natural & SOFT (no processing that hardens it) ----------
config.ENABLE_VOICE_ENHANCE = False   # no denoiser altering the clone
config.ENABLE_AUDIO_CLEANUP = False   # no loudness compression (keeps softness)

config.PRESERVE_BACKGROUND = True     # keep original music/SFX
config.GENERATE_SUBTITLES = True

print('Device:', config.DEVICE, '| TTS engine:', config.TTS_BACKEND,
      '| enhance:', config.ENABLE_VOICE_ENHANCE,
      '| mastering:', config.ENABLE_AUDIO_CLEANUP)

## 5. Upload your video and a reference voice

- **Video:** the MP4 you want to dub / revoice.
- **Reference voice:** a clean 15-30s clip (MP3/WAV) of the voice to clone.

In [ ]:
from google.colab import files

print('Select your VIDEO file...')
video_path = list(files.upload().keys())[0]
print('Select your REFERENCE VOICE file...')
reference_path = list(files.upload().keys())[0]
print('Video:', video_path, '| Reference:', reference_path)

## 6. Dub it

Set `TARGET` to a language code to translate+dub, or leave it `None` to just
replace the voice in the original language.

Codes: en, es, fr, de, it, pt, pl, tr, ru, nl, cs, ar, zh-cn, ja, hu, ko, hi

In [ ]:
import core

TARGET = None   # e.g. 'es' for Spanish, 'hi' for Hindi; None = keep language
SOURCE = 'auto' # or a specific code if auto-detect guesses wrong

out = core.process_video(
    video_path,
    reference_path=reference_path,
    source_lang=SOURCE,
    target_lang=TARGET,
    cb=print,
)
print('\nDONE ->', out)

## 7. Download the result

In [ ]:
from google.colab import files
from pathlib import Path

# Download the final video plus any subtitle files written next to it.
files.download(out)
stem = Path(out).with_suffix('')
for ext in ('.srt', '.vtt'):
    p = f'{stem}{ext}'
    if Path(p).is_file():
        files.download(p)